# 04 - Relaciones entre bases ENIGH

Objetivo: documentar la granularidad de las tablas principales, validar llaves y cardinalidades con datos reales, y dejar claro cómo evitar multiplicaciones de filas antes de construir bases analíticas.

Nota de ruta: en el proyecto local los archivos están bajo `data/raw/EINGH/`, aunque en algunas instrucciones aparece `ENIGH/`.

In [3]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

try:
    from IPython.display import display, Markdown
except ImportError:
    display = print
    Markdown = lambda text: text


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("No pude localizar la raíz del proyecto.")


ROOT = find_project_root()
RAW = ROOT / "data" / "raw" / "EINGH"
REV3 = ROOT / "data" / "interim" / "revision_3"
REV4 = ROOT / "data" / "interim" / "revision_4"
REV4.mkdir(parents=True, exist_ok=True)

YEARS = [2018, 2020, 2022, 2024]
TABLES = ["viviendas", "hogares", "concentradohogar", "poblacion", "trabajos", "ingresos"]

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

print(f"Proyecto: {ROOT}")
print(f"Datos crudos: {RAW}")

Proyecto: c:\Users\lucia\OneDrive\Escritorio\Fer\inegi-income-modeling
Datos crudos: c:\Users\lucia\OneDrive\Escritorio\Fer\inegi-income-modeling\data\raw\EINGH


## 1. Evidencia documental

Se utiliza el inventario de metadatos extraído de los PDFs oficiales y se verifica que los PDFs locales existan.

In [4]:
metadata = pd.read_csv(ROOT / "docs" / "enigh_variable_metadata.csv")
pdfs = pd.DataFrame(
    {
        "anio": YEARS,
        "pdf_local": [RAW / str(year) / f"doc_{year}.pdf" for year in YEARS],
    }
)
pdfs["existe"] = pdfs["pdf_local"].map(lambda p: p.exists())
display(pdfs)

key_vars = ["folioviv", "foliohog", "numren", "id_trabajo", "clave", "mes_1", "ing_1", "ing_tri", "factor", "est_dis", "upm"]
doc_keys = (
    metadata[
        metadata["table"].isin([f"{table}.csv" for table in TABLES])
        & metadata["variable"].isin(key_vars)
    ][["year", "table", "position", "variable", "label", "dtype", "source_page"]]
    .sort_values(["year", "table", "position"])
)
display(doc_keys.head(80))

,anio,pdf_local,existe
0,2018,c:\Users\lucia\OneDrive\Escritorio\Fer\inegi-income-modeling\data\raw\EINGH\2018\doc_2018.pdf,True
1,2020,c:\Users\lucia\OneDrive\Escritorio\Fer\inegi-income-modeling\data\raw\EINGH\2020\doc_2020.pdf,True
2,2022,c:\Users\lucia\OneDrive\Escritorio\Fer\inegi-income-modeling\data\raw\EINGH\2022\doc_2022.pdf,True
3,2024,c:\Users\lucia\OneDrive\Escritorio\Fer\inegi-income-modeling\data\raw\EINGH\2024\doc_2024.pdf,True


,year,table,position,variable,label,dtype,source_page
682,2018,concentradohogar.csv,1,folioviv,Identificador de la vivienda,C (10),28
683,2018,concentradohogar.csv,2,foliohog,Identificador del hogar,C (1),28
687,2018,concentradohogar.csv,6,est_dis,Estrato de diseño muestral,C (7),28
688,2018,concentradohogar.csv,7,upm,Unidad primaria de muestreo,C (5),28
689,2018,concentradohogar.csv,8,factor,Factor de expansión,N (5),28
...,...,...,...,...,...,...,...
2112,2022,poblacion.csv,188,factor,Factor de expansión,N (5),24
2211,2022,trabajos.csv,1,folioviv,Identificador de la vivienda,C (10),26
2212,2022,trabajos.csv,2,foliohog,Identificador del hogar,C (1),26
2213,2022,trabajos.csv,3,numren,Identificador de la persona,C (2),26


## 2. Mapa relacional propuesto

Esta tabla no se toma como verdad por diseño; se valida en las siguientes celdas con cardinalidades empíricas.

In [5]:
mapa_relacional = pd.DataFrame(
    [
        ["viviendas", "vivienda", "folioviv", "1:N hogares"],
        ["hogares", "hogar", "folioviv + foliohog", "N:1 vivienda; 1:N poblacion; 1:1 concentradohogar"],
        ["concentradohogar", "hogar agregado INEGI", "folioviv + foliohog", "1:1 hogares"],
        ["poblacion", "persona", "folioviv + foliohog + numren", "N:1 hogar; 1:N trabajos; 1:N ingresos"],
        ["trabajos", "trabajo de una persona", "folioviv + foliohog + numren + id_trabajo", "N:1 persona"],
        ["ingresos", "registro persona-clave de ingreso", "folioviv + foliohog + numren + clave", "N:1 persona"],
    ],
    columns=["tabla", "nivel", "llave", "relacion_principal"],
)
mapa_relacional.to_csv(REV4 / "mapa_relacional.csv", index=False, encoding="utf-8")
display(mapa_relacional)

,tabla,nivel,llave,relacion_principal
0,viviendas,vivienda,folioviv,1:N hogares
1,hogares,hogar,folioviv + foliohog,N:1 vivienda; 1:N poblacion; 1:1 concentradohogar
2,concentradohogar,hogar agregado INEGI,folioviv + foliohog,1:1 hogares
3,poblacion,persona,folioviv + foliohog + numren,N:1 hogar; 1:N trabajos; 1:N ingresos
4,trabajos,trabajo de una persona,folioviv + foliohog + numren + id_trabajo,N:1 persona
5,ingresos,registro persona-clave de ingreso,folioviv + foliohog + numren + clave,N:1 persona


## 3. Validación de cardinalidades

Se valida cada llave propuesta por año. Una llave válida debe tener 0 filas duplicadas.

In [6]:
KEYS = {
    "viviendas": ["folioviv"],
    "hogares": ["folioviv", "foliohog"],
    "concentradohogar": ["folioviv", "foliohog"],
    "poblacion": ["folioviv", "foliohog", "numren"],
    "trabajos": ["folioviv", "foliohog", "numren", "id_trabajo"],
    "ingresos": ["folioviv", "foliohog", "numren", "clave"],
}
PARENT_KEYS = {
    "hogares": ["folioviv"],
    "concentradohogar": ["folioviv", "foliohog"],
    "poblacion": ["folioviv", "foliohog"],
    "trabajos": ["folioviv", "foliohog", "numren"],
    "ingresos": ["folioviv", "foliohog", "numren"],
}


def read_raw(year, table, usecols=None):
    return pd.read_csv(RAW / str(year) / f"{table}.csv", usecols=usecols, dtype=str, low_memory=False)


rows = []
for year in YEARS:
    for table in TABLES:
        key = KEYS[table]
        parent_key = PARENT_KEYS.get(table)
        usecols = sorted(set(key + (parent_key or [])))
        df = read_raw(year, table, usecols=usecols)
        filas = len(df)
        llaves_unicas = df.drop_duplicates(key).shape[0]
        rows.append(
            {
                "anio": year,
                "tabla": table,
                "filas": filas,
                "llave_propuesta": " + ".join(key),
                "llaves_unicas": llaves_unicas,
                "filas_duplicadas_llave": filas - llaves_unicas,
                "pct_filas_duplicadas_llave": round((filas - llaves_unicas) / filas * 100, 4) if filas else 0,
                "unidad_superior": " + ".join(parent_key) if parent_key else "",
                "max_registros_por_unidad_superior": int(df.groupby(parent_key, dropna=False).size().max()) if parent_key else 1,
            }
        )

cardinalidades = pd.DataFrame(rows)
cardinalidades.to_csv(REV4 / "validacion_cardinalidades.csv", index=False, encoding="utf-8")
display(cardinalidades)
assert cardinalidades["filas_duplicadas_llave"].sum() == 0

,anio,tabla,filas,llave_propuesta,llaves_unicas,filas_duplicadas_llave,pct_filas_duplicadas_llave,unidad_superior,max_registros_por_unidad_superior
0,2018,viviendas,73405,folioviv,73405,0,0.0,,1
1,2018,hogares,74647,folioviv + foliohog,74647,0,0.0,folioviv,5
2,2018,concentradohogar,74647,folioviv + foliohog,74647,0,0.0,folioviv + foliohog,1
3,2018,poblacion,269206,folioviv + foliohog + numren,269206,0,0.0,folioviv + foliohog,22
4,2018,trabajos,139933,folioviv + foliohog + numren + id_trabajo,139933,0,0.0,folioviv + foliohog + numren,2
5,2018,ingresos,348487,folioviv + foliohog + numren + clave,348487,0,0.0,folioviv + foliohog + numren,13
6,2020,viviendas,87754,folioviv,87754,0,0.0,,1
7,2020,hogares,89006,folioviv + foliohog,89006,0,0.0,folioviv,5
8,2020,concentradohogar,89006,folioviv + foliohog,89006,0,0.0,folioviv + foliohog,1
9,2020,poblacion,315743,folioviv + foliohog + numren,315743,0,0.0,folioviv + foliohog,25


## 4. Validación de relaciones entre tablas

Se comprueba que cada unidad hija tenga padre y que las relaciones potencialmente 1:N se entiendan antes de hacer merges.

In [7]:
relationship_rows = []
for year in YEARS:
    viviendas = read_raw(year, "viviendas", ["folioviv"])
    hogares = read_raw(year, "hogares", ["folioviv", "foliohog"])
    concentrado = read_raw(year, "concentradohogar", ["folioviv", "foliohog"])
    poblacion = read_raw(year, "poblacion", ["folioviv", "foliohog", "numren"])
    trabajos = read_raw(year, "trabajos", ["folioviv", "foliohog", "numren", "id_trabajo"])
    ingresos = read_raw(year, "ingresos", ["folioviv", "foliohog", "numren", "clave"])

    checks = [
        ("viviendas -> hogares", viviendas.drop_duplicates(), hogares.drop_duplicates(), ["folioviv"], "1:N"),
        ("hogares -> concentradohogar", hogares.drop_duplicates(), concentrado.drop_duplicates(), ["folioviv", "foliohog"], "1:1"),
        ("hogares -> poblacion", hogares.drop_duplicates(), poblacion.drop_duplicates(), ["folioviv", "foliohog"], "1:N"),
        ("poblacion -> trabajos", poblacion.drop_duplicates(), trabajos.drop_duplicates(), ["folioviv", "foliohog", "numren"], "1:N opcional"),
        ("poblacion -> ingresos", poblacion.drop_duplicates(), ingresos.drop_duplicates(), ["folioviv", "foliohog", "numren"], "1:N opcional"),
    ]

    for relacion, parent, child, key, tipo in checks:
        merged_child = child.merge(parent[key].drop_duplicates().assign(_parent_match=1), on=key, how="left")
        merged_parent = parent.merge(child[key].drop_duplicates().assign(_child_match=1), on=key, how="left")
        child_counts = child.groupby(key, dropna=False).size()
        relationship_rows.append(
            {
                "anio": year,
                "relacion": relacion,
                "tipo_esperado": tipo,
                "llave": " + ".join(key),
                "unidades_padre": len(parent),
                "unidades_hijas": len(child),
                "sin_padre": int(merged_child["_parent_match"].isna().sum()),
                "pct_hijas_con_padre": round(merged_child["_parent_match"].notna().mean() * 100, 4),
                "padres_sin_hijas": int(merged_parent["_child_match"].isna().sum()),
                "pct_padres_con_hijas": round(merged_parent["_child_match"].notna().mean() * 100, 4),
                "max_hijos_por_padre": int(child_counts.max()),
            }
        )

relaciones = pd.DataFrame(relationship_rows)
relaciones.to_csv(REV4 / "validacion_relaciones.csv", index=False, encoding="utf-8")
display(relaciones)
assert relaciones["sin_padre"].sum() == 0

,anio,relacion,tipo_esperado,llave,unidades_padre,unidades_hijas,sin_padre,pct_hijas_con_padre,padres_sin_hijas,pct_padres_con_hijas,max_hijos_por_padre
0,2018,viviendas -> hogares,1:N,folioviv,73405,74647,0,100.0,0,100.0000,5
1,2018,hogares -> concentradohogar,1:1,folioviv + foliohog,74647,74647,0,100.0,0,100.0000,1
2,2018,hogares -> poblacion,1:N,folioviv + foliohog,74647,269206,0,100.0,0,100.0000,22
3,2018,poblacion -> trabajos,1:N opcional,folioviv + foliohog + numren,269206,139933,0,100.0,141568,47.4128,2
4,2018,poblacion -> ingresos,1:N opcional,folioviv + foliohog + numren,269206,348487,0,100.0,86772,67.7674,13
5,2020,viviendas -> hogares,1:N,folioviv,87754,89006,0,100.0,0,100.0000,5
6,2020,hogares -> concentradohogar,1:1,folioviv + foliohog,89006,89006,0,100.0,0,100.0000,1
7,2020,hogares -> poblacion,1:N,folioviv + foliohog,89006,315743,0,100.0,0,100.0000,25
8,2020,poblacion -> trabajos,1:N opcional,folioviv + foliohog + numren,315743,164876,0,100.0,166356,47.3128,2
9,2020,poblacion -> ingresos,1:N opcional,folioviv + foliohog + numren,315743,394912,0,100.0,111411,64.7147,12


## 5. Diagrama relacional

```mermaid
erDiagram
    VIVIENDAS ||--o{ HOGARES : folioviv
    HOGARES ||--|| CONCENTRADOHOGAR : folioviv_foliohog
    HOGARES ||--o{ POBLACION : folioviv_foliohog
    POBLACION ||--o{ TRABAJOS : persona
    POBLACION ||--o{ INGRESOS : persona
```


## 6. Ingresos

Cada fila de `ingresos` es una combinación persona-clave de ingreso. `clave` identifica el concepto y `ing_tri` reporta ingreso trimestral. Antes de unir a persona u hogar, se agregan las filas a la granularidad correspondiente.

In [8]:
ing = pd.read_csv(
    REV3 / "ingresos_decodificada_2018_2024.csv.gz",
    usecols=["anio", "folioviv", "foliohog", "numren", "clave", "clave_desc", "ing_tri"],
    dtype=str,
    low_memory=False,
)
ing["ing_tri_num"] = pd.to_numeric(ing["ing_tri"], errors="coerce").fillna(0)
ing["clave_num"] = pd.to_numeric(ing["clave"].str.replace("P", "", regex=False), errors="coerce")

ingresos_persona = (
    ing.groupby(["anio", "folioviv", "foliohog", "numren"], dropna=False)
    .agg(
        registros_ingreso=("clave", "size"),
        claves_ingreso_distintas=("clave", "nunique"),
        ingreso_persona_total_registros_tri=("ing_tri_num", "sum"),
    )
    .reset_index()
)
ingresos_persona.to_csv(REV4 / "ingresos_agregados_persona.csv.gz", index=False, compression="gzip")

resumen_ing = (
    ingresos_persona.groupby("anio")
    .agg(
        personas_con_ingreso=("numren", "size"),
        mediana_registros=("registros_ingreso", "median"),
        max_registros=("registros_ingreso", "max"),
        mediana_ingreso=("ingreso_persona_total_registros_tri", "median"),
    )
    .reset_index()
)
display(resumen_ing)

,anio,personas_con_ingreso,mediana_registros,max_registros,mediana_ingreso
0,2018,182434,2.0,13,10173.91
1,2020,204332,2.0,12,11739.13
2,2022,205144,2.0,13,16141.30
3,2024,202365,2.0,14,20543.47


La suma de `ingresos.csv` se conserva como derivado persona-clave. Para ingreso oficial de hogar se priorizan las variables ya agregadas de `concentradohogar`, especialmente `ing_cor` e `ingtrab`.

## 7. Trabajos

Cada fila de `trabajos` corresponde a un trabajo de una persona. La llave empírica es `folioviv + foliohog + numren + id_trabajo`; el máximo observado es 2 trabajos por persona.

In [ ]:
trab = pd.read_csv(
    REV3 / "trabajos_decodificada_2018_2024.csv.gz",
    usecols=["anio", "folioviv", "foliohog", "numren", "id_trabajo", "htrab", "pago", "pago_desc", "contrato", "contrato_desc"],
    dtype=str,
    low_memory=False,
)
trab["htrab_num"] = pd.to_numeric(trab["htrab"], errors="coerce")
trab["id_trabajo_num"] = pd.to_numeric(trab["id_trabajo"], errors="coerce")
trab = trab.sort_values(["anio", "folioviv", "foliohog", "numren", "id_trabajo_num"], na_position="last")

trabajos_persona = (
    trab.groupby(["anio", "folioviv", "foliohog", "numren"], dropna=False)
    .agg(
        n_trabajos=("id_trabajo", "size"),
        horas_trabajos_total=("htrab_num", "sum"),
        horas_trabajo_principal=("htrab_num", "first"),
        id_trabajo_principal=("id_trabajo", "first"),
        pago_principal_desc=("pago_desc", "first"),
        contrato_principal_desc=("contrato_desc", "first"),
    )
    .reset_index()
)

display(trabajos_persona.groupby("anio").agg(
    personas_con_trabajo    =("numren", "size"), 
    max_trabajos            =("n_trabajos", "max"), 
    mediana_horas           =("horas_trabajos_total", "median")
    ).reset_index())

,anio,personas_con_trabajo,max_trabajos,mediana_horas
0,2018,127638,2,48.0
1,2020,149387,2,48.0
2,2022,150682,2,48.0
3,2024,150382,2,48.0


## 8. Conclusiones metodológicas

- Las llaves propuestas son únicas en los cuatro años.
- `hogares` y `concentradohogar` se relacionan 1:1 por `folioviv + foliohog`.
- `poblacion`, `trabajos` e `ingresos` no deben unirse directamente sin agregación previa.
- `trabajos` debe reducirse primero a persona para evitar duplicar personas con dos trabajos.
- `ingresos` debe reducirse primero a persona-clave o persona, porque una persona puede tener hasta 14 claves observadas.
- La geografía se incorpora de forma segura desde hogar/vivienda; municipio es útil para exploración, no para inferencia municipal automática.
